# Module 5 – Advanced Applications and Production Deployment
### Course: OpenAI Embeddings API | Pluralsight
**Author:** Harit Himanshu  ·  **Version:** After (Complete Solution)

In [ ]:
%pip install -q openai pinecone numpy scikit-learn matplotlib umap-learn

---
## Clip 1: Building a RAG System with Embeddings

In [ ]:
import os, json, getpass
from openai import OpenAI
from pinecone import Pinecone

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter Pinecone API key: ")
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

with open("../module2.json") as f:
    embedded_reviews = json.load(f)

INDEX_NAME = "yelp-restaurant-finder"
NAMESPACE  = "yelp-reviews"
index = pc.Index(host=pc.describe_index(INDEX_NAME).host)

print(f"Reviews: {len(embedded_reviews)}, Index: {INDEX_NAME}")

In [ ]:
def chunk_fixed_size(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into fixed-size word chunks with overlap."""
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def chunk_by_sentences(text: str, max_sentences: int = 5) -> list[str]:
    """Split text into chunks of max_sentences sentences."""
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = ". ".join(sentences[i:i + max_sentences]) + "."
        chunks.append(chunk)
    return chunks


long_review = ("This place is absolutely fantastic. " * 40 +
               "The food quality has improved since last year. " * 30 +
               "Service is always prompt and friendly. " * 20)

print(f"Original: {len(long_review.split())} words")

fixed_chunks = chunk_fixed_size(long_review, chunk_size=200, overlap=20)
print(f"Fixed-size chunks (200 words, 20 overlap): {len(fixed_chunks)} chunks")
print(f"  Chunk 0 preview: {fixed_chunks[0][:60]}...")

sentence_chunks = chunk_by_sentences(long_review, max_sentences=3)
print(f"Sentence chunks (3 sentences each): {len(sentence_chunks)} chunks")

In [ ]:
SYSTEM_PROMPT = """You are a helpful restaurant concierge for an Intelligent Restaurant Finder.
Answer questions about restaurants using ONLY the review excerpts provided as context.
If the context does not contain relevant information, say so honestly.
Be concise, friendly, and specific — reference details from the reviews."""


def rag_answer(question: str, top_k: int = 5) -> str:
    """
    Full RAG pipeline using OpenAI embeddings + Pinecone + GPT-4o-mini.
    Ref: https://platform.openai.com/docs/api-reference/chat/create
    """
    # Step 1: Embed the question
    embed_response = client.embeddings.create(
        input=question,
        model="text-embedding-3-small"
    )
    query_vec = embed_response.data[0].embedding

    # Step 2: Retrieve top_k relevant reviews from Pinecone
    results = index.query(
        vector=query_vec,
        top_k=top_k,
        include_metadata=True,
        namespace=NAMESPACE
    )

    # Step 3: Format retrieved context
    context_parts = []
    for i, match in enumerate(results.matches, 1):
        text = match.metadata.get("text", "")
        stars = match.metadata.get("stars", "?")
        city = match.metadata.get("city", "Unknown")
        context_parts.append(f"Review {i} ({stars}★, {city}): {text}")
    context = "\n\n".join(context_parts)

    # Step 4: Call Chat Completions API
    # Model: gpt-4o-mini (cost-efficient, fast for RAG tasks)
    chat_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0.3,
        max_tokens=300
    )

    return chat_response.choices[0].message.content


## TODO: #1 - Call rag_answer() with a sample question and print the answer.


---
## Clip 2: Clustering, Classification, and Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import umap

embeddings_matrix = np.array([r["embedding"] for r in embedded_reviews])
print(f"Embeddings matrix: {embeddings_matrix.shape}")

In [ ]:
# K-MEANS CLUSTERING
# Ref: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

K = 5
embeddings_matrix = np.array([r["embedding"] for r in embedded_reviews])
texts = [r["text"] for r in embedded_reviews]

print(f"Fitting KMeans with K={K} on {embeddings_matrix.shape}...")
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings_matrix)

# Silhouette score
sil_score = silhouette_score(embeddings_matrix, cluster_labels, sample_size=min(500, len(embedded_reviews)))
print(f"Silhouette Score: {sil_score:.3f}  (closer to 1.0 = better-defined clusters)")

# Show representative reviews per cluster (closest to centroid)
print("\n--- Cluster Representatives ---")
for k in range(K):
    cluster_indices = np.where(cluster_labels == k)[0]
    centroid = kmeans.cluster_centers_[k]
    distances = np.linalg.norm(embeddings_matrix[cluster_indices] - centroid, axis=1)
    top3 = cluster_indices[np.argsort(distances)[:3]]
    print(f"\nCluster {k} ({len(cluster_indices)} reviews):")
    for idx in top3:
        print(f"  - [{embedded_reviews[idx]['stars']}★] {texts[idx][:80]}...")

In [ ]:
# STAR RATING CLASSIFIER (KNN on embeddings)
# Ref: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

X = embeddings_matrix
y = np.array([r["stars"] for r in embedded_reviews])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print("KNN Star Rating Classifier:")
print(classification_report(y_test, y_pred, labels=[1,2,3,4,5]))

# Predict star rating for a new review
test_reviews = [
    "Absolutely incredible food and service. Best meal of my life!",
    "Terrible experience. Cold food, rude staff, never coming back.",
    "Pretty decent, nothing special but solid overall.",
]
print("\nPredicting star ratings for new reviews:")
for rev in test_reviews:
    vec = np.array(client.embeddings.create(input=rev, model="text-embedding-3-small").data[0].embedding)
    pred = knn.predict([vec])[0]
    print(f"  [{pred}★] {rev[:60]}...")

In [ ]:
# t-SNE VISUALIZATION
# Ref: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html

SAMPLE_SIZE = 200
sample_idx = np.random.choice(len(embedded_reviews), SAMPLE_SIZE, replace=False)
sample_embeddings = embeddings_matrix[sample_idx]
sample_stars = np.array([embedded_reviews[i]["stars"] for i in sample_idx])

print(f"Fitting t-SNE on {SAMPLE_SIZE} reviews...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_2d = tsne.fit_transform(sample_embeddings)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    tsne_2d[:, 0], tsne_2d[:, 1],
    c=sample_stars, cmap="RdYlGn",
    alpha=0.75, s=40, edgecolors="none"
)
plt.colorbar(scatter, label="Star Rating")
plt.title(f"t-SNE of Yelp Review Embeddings (n={SAMPLE_SIZE}, colored by star rating)")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.tight_layout()
plt.savefig("tsne_yelp_embeddings.png", dpi=120)
plt.show()
print("Saved: tsne_yelp_embeddings.png")

In [ ]:
# UMAP VISUALIZATION — side-by-side comparison with t-SNE
# Ref: https://umap-learn.readthedocs.io/en/latest/basic_usage.html

print("Fitting UMAP on same sample...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)
umap_2d = reducer.fit_transform(sample_embeddings)

# Side-by-side plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, coords, title in [
    (axes[0], tsne_2d, "t-SNE"),
    (axes[1], umap_2d, "UMAP")
]:
    sc = ax.scatter(
        coords[:, 0], coords[:, 1],
        c=sample_stars, cmap="RdYlGn",
        alpha=0.75, s=40, edgecolors="none"
    )
    plt.colorbar(sc, ax=ax, label="Stars")
    ax.set_title(f"{title} — Yelp Review Embeddings\n(n={SAMPLE_SIZE}, colored by star rating)")
    ax.set_xlabel(f"{title} Dim 1")
    ax.set_ylabel(f"{title} Dim 2")

plt.tight_layout()
plt.savefig("tsne_umap_comparison.png", dpi=120)
plt.show()
print("Saved: tsne_umap_comparison.png")
print("\nObservation: UMAP typically preserves global structure better and runs faster.")